# Neuronale Netze mit NumPy: ein einfacher Einstieg

Dieses Notebook ist eine **komplett überarbeitete** und **deutschsprachige** Einführung in kleine neuronale Netze mit NumPy.

Der Fokus liegt auf drei Punkten:

1. **Verstehen**, wie ein kleines neuronales Netz aufgebaut ist.
2. **Korrekt implementieren**, wie Gradienten mit **automatischer Differenzierung** berechnet werden.
3. Das Ganze an einem einfachen Beispiel trainieren: Wir approximieren die Funktion $\sin(x)$.

Wir arbeiten bewusst **klein, langsam und transparent**.  
Das Ziel ist **nicht** maximale Geschwindigkeit, sondern ein sauberer Einstieg für Studierende.

## Was Sie nach diesem Notebook können sollten

Nach dem Durcharbeiten sollten Sie erklären können,

- was ein Parameter in einem neuronalen Netz ist,
- was bei einem Vorwärtsdurchlauf passiert,
- warum man für das Training Ableitungen braucht,
- wie die **Kettenregel** in einem Rechengraphen benutzt wird,
- und wie ein einfaches Netz mit Gradientenverfahren trainiert wird.

Wichtig: Unsere Implementierung ist zum Lernen der Konzepte.  
Bibliotheken wie PyTorch oder JAX sind deutlich leistungsfähiger. Hier bauen wir die Kernideen selbst nach.


## Das mathematische Grundproblem

Wir betrachten überwachte Lernaufgaben.  
Gegeben sind Eingaben $x_1,\dots,x_N$ und dazugehörige Zielwerte $y_1,\dots,y_N$.  
Gesucht ist eine Funktion $F(\,\cdot\,;\theta)$, die von Parametern $\theta$ abhängt und die Daten möglichst gut approximiert.

Mathematisch formulieren wir das als Minimierungsproblem:
$$
\min_{\theta}\; \frac{1}{N}\sum_{i=1}^N \ell\bigl(F(x_i;\theta),\,y_i\bigr).
$$

Dabei sind

- $F(x_i;\theta)$ die Vorhersagen des Modells,
- $\ell$ die Verlustfunktion,
- $\theta$ alle trainierbaren Gewichte und Bias-Terme.

In diesem Notebook verwenden wir als Trainingsbeispiel die Approximation der Funktion
$$
f(x)=\sin(x).
$$
Das Ziel ist also nicht nur „ein Netz zu programmieren“, sondern ein konkretes Optimierungsproblem numerisch zu lösen.

Warum sind Ableitungen hier so wichtig?  
Weil wir die Parameter $\theta$ mit einem Gradientenverfahren aktualisieren:
$$
\theta^{(k+1)}=\theta^{(k)}-\eta \nabla_\theta L(\theta^{(k)}),
$$
wobei $\eta>0$ die Lernrate ist und $L$ die gesamte Verlustfunktion bezeichnet.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Das Lernbeispiel

Wir wollen aus Daten die Funktion $\sin(x)$ lernen.

Warum ist das ein gutes Beispiel?

- Die Funktion ist bekannt.
- Man sieht sofort, ob die Vorhersage sinnvoll ist.
- Das Ziel ist leicht verständlich: Das Netz soll eine Kurve nachzeichnen.

In [ ]:
x_train = np.linspace(-np.pi, np.pi, 256).reshape(-1, 1)
y_train = np.sin(x_train)

dx = x_train[1, 0] - x_train[0, 0]
x_val = (np.linspace(-np.pi, np.pi, 256) + 0.5 * dx).reshape(-1, 1)
y_val = np.sin(x_val)

plt.figure(figsize=(7, 4))
plt.plot(x_train, y_train, label="Trainingsdaten: sin(x)")
plt.scatter(x_val[::8], y_val[::8], s=12, label="Validierungsdaten", alpha=0.7)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Datensatz")
plt.legend()
plt.show()


## Das Lernziel in Formeln

Ein einfaches voll verbundenes neuronales Netz besteht aus wiederholten Schritten der Form
$$
x \mapsto xW+b
\quad\text{und danach}\quad
x \mapsto \sigma(x),
$$
also aus einer linearen Abbildung und einer nichtlinearen Aktivierungsfunktion.

Für ein Netz mit einer versteckten Schicht könnte man zum Beispiel schreiben:
$$
Z^{(1)} = XW^{(1)} + b^{(1)},\qquad
A^{(1)} = \tanh\!\bigl(Z^{(1)}\bigr),
$$
$$
\widehat{Y} = A^{(1)}W^{(2)} + b^{(2)}.
$$

Hierbei bedeutet:

- $X$: Eingabedaten eines Mini-Batches,
- $W^{(1)},W^{(2)}$: Gewichtsmatrizen,
- $b^{(1)},b^{(2)}$: Bias-Terme,
- $\widehat{Y}$: Vorhersage des Netzes.

Im späteren Code verwenden wir ein etwas tieferes Netz.  
Die mathematische Idee bleibt aber genau dieselbe: Verkettung einfacher Bausteine.

Als Verlustfunktion wählen wir den mittleren quadratischen Fehler:
$$
L(\widehat{Y},Y)=\frac{1}{m}\sum_{i=1}^m \bigl(\widehat{y}_i-y_i\bigr)^2,
$$
wobei $m$ die Batchgröße ist.


## Idee der automatischen Differenzierung

Wir bauen eine kleine Klasse `Tensor`.

Jedes Objekt speichert

- den **numerischen Wert** `data`,
- den **Gradienten** `grad`,
- und Informationen darüber, **aus welchen vorherigen Objekten** es entstanden ist.

Wenn wir am Ende einen skalaren Fehlerwert `loss` haben, rufen wir `loss.backward()` auf.

Dann passiert Folgendes:

1. Wir laufen den Rechengraphen **rückwärts** durch.
2. An jedem Knoten wenden wir die **lokale Ableitungsregel** an.
3. Die Gradienten werden mit der **Kettenregel** zusammengesetzt.

Genau das ist automatische Differenzierung.


## Warum automatische Differenzierung funktioniert

Die zentrale Regel ist die **Kettenregel**.  
Wenn eine Größe $u$ von $v$ abhängt und $v$ wiederum von $x$, dann gilt
$$
\frac{\mathrm d u}{\mathrm d x}
=
\frac{\mathrm d u}{\mathrm d v}\cdot
\frac{\mathrm d v}{\mathrm d x}.
$$

Ein Rechenweg wie
$$
x \longrightarrow z \longrightarrow a \longrightarrow L
$$
führt daher auf
$$
\frac{\partial L}{\partial x}
=
\frac{\partial L}{\partial a}
\frac{\partial a}{\partial z}
\frac{\partial z}{\partial x}.
$$

Genau das macht automatische Differenzierung:

1. **Vorwärtslauf:** Alle Zwischenwerte werden ausgerechnet.
2. **Rückwärtslauf:** Mit der Kettenregel werden die Gradienten Schritt für Schritt zurückpropagiert.

Wichtig ist: Wir differenzieren **nicht symbolisch** und wir approximieren die Ableitungen auch **nicht numerisch**.  
Stattdessen nutzen wir die tatsächlich ausgeführten Rechenschritte und die lokalen Ableitungen dieser einzelnen Operationen.



## Ein Beispiel mit nur einem Neuron

Bevor wir allgemeine Klassen programmieren, lohnt sich ein Blick auf das einfachste Beispiel:
$$
z = wx+b,\qquad a=\tanh(z),\qquad L=\frac12(a-y)^2.
$$

Dann erhält man mit der Kettenregel
$$
\frac{\partial L}{\partial a}=a-y,
$$
$$
\frac{\partial a}{\partial z}=1-\tanh^2(z)=1-a^2,
$$
also
$$
\frac{\partial L}{\partial z}
=
(a-y)(1-a^2).
$$

Damit folgen die Ableitungen nach den Parametern:
$$
\frac{\partial L}{\partial w}
=
\frac{\partial L}{\partial z}\,x,
\qquad
\frac{\partial L}{\partial b}
=
\frac{\partial L}{\partial z}.
$$

Diese Formeln sind der Kern der Backpropagation.  
Bei größeren Netzen passiert konzeptionell nichts Neues — es gibt nur mehr Zwischenstufen.


## Eine kleine technische Hilfsfunktion

Bei Operationen wie

$$
XW + b
$$

wird der Bias $b$ über viele Zeilen hinweg **broadcastet**.  
Im Rückwärtslauf muss der Gradient daher wieder auf die richtige Form zurückgeführt werden.

Diese Hilfsfunktion erledigt genau das.

In [ ]:
def sum_to_shape(grad, shape):
    """Summiert einen Gradienten so, dass seine Form wieder zu `shape` passt.

    Das ist wichtig für Broadcasting, zum Beispiel bei `X @ W + b`.
    """
    grad = np.asarray(grad, dtype=float)

    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)

    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)

    return grad

## Die Klasse `Tensor`

Diese Klasse ist das Herzstück des Notebooks.

Wir implementieren nur wenige Operationen:

- Addition
- Subtraktion
- elementweise Multiplikation
- Matrixmultiplikation
- Potenzen
- Summe und Mittelwert
- Aktivierungsfunktionen `relu` und `tanh`

Das reicht bereits für ein kleines neuronales Netz.

In [ ]:
class Tensor:
    def __init__(self, data, requires_grad=False, _children=(), _op=""):
        self.data = np.array(data, dtype=float)
        self.requires_grad = requires_grad
        self.grad = np.zeros_like(self.data) if self.requires_grad else None

        self._prev = set(_children)
        self._op = _op
        self._backward = lambda: None

    def __repr__(self):
        return f"Tensor(data={self.data}, grad={self.grad})"

    def zero_grad(self):
        if self.requires_grad:
            self.grad = np.zeros_like(self.data)

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)

        out = Tensor(
            self.data + other.data,
            requires_grad=self.requires_grad or other.requires_grad,
            _children=(self, other),
            _op="+",
        )

        def _backward():
            if self.requires_grad:
                self.grad += sum_to_shape(out.grad, self.data.shape)
            if other.requires_grad:
                other.grad += sum_to_shape(out.grad, other.data.shape)

        out._backward = _backward
        return out

    __radd__ = __add__

    def __neg__(self):
        out = Tensor(-self.data, requires_grad=self.requires_grad, _children=(self,), _op="neg")

        def _backward():
            if self.requires_grad:
                self.grad -= out.grad

        out._backward = _backward
        return out

    def __sub__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        return self + (-other)

    def __rsub__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        return other + (-self)

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)

        out = Tensor(
            self.data * other.data,
            requires_grad=self.requires_grad or other.requires_grad,
            _children=(self, other),
            _op="*",
        )

        def _backward():
            if self.requires_grad:
                self.grad += sum_to_shape(out.grad * other.data, self.data.shape)
            if other.requires_grad:
                other.grad += sum_to_shape(out.grad * self.data, other.data.shape)

        out._backward = _backward
        return out

    __rmul__ = __mul__

    def pow(self, exponent):
        out = Tensor(
            self.data ** exponent,
            requires_grad=self.requires_grad,
            _children=(self,),
            _op=f"pow({exponent})",
        )

        def _backward():
            if self.requires_grad:
                self.grad += out.grad * exponent * (self.data ** (exponent - 1))

        out._backward = _backward
        return out

    def __truediv__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        return self * other.pow(-1)

    def __rtruediv__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        return other / self

    def __matmul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)

        out = Tensor(
            self.data @ other.data,
            requires_grad=self.requires_grad or other.requires_grad,
            _children=(self, other),
            _op="@",
        )

        def _backward():
            if self.requires_grad:
                self.grad += out.grad @ other.data.T
            if other.requires_grad:
                other.grad += self.data.T @ out.grad

        out._backward = _backward
        return out

    @property
    def T(self):
        out = Tensor(self.data.T, requires_grad=self.requires_grad, _children=(self,), _op="transpose")

        def _backward():
            if self.requires_grad:
                self.grad += out.grad.T

        out._backward = _backward
        return out

    def sum(self, axis=None, keepdims=False):
        out = Tensor(
            self.data.sum(axis=axis, keepdims=keepdims),
            requires_grad=self.requires_grad,
            _children=(self,),
            _op="sum",
        )

        def _backward():
            if not self.requires_grad:
                return

            grad = out.grad

            if axis is None:
                self.grad += np.ones_like(self.data) * grad
                return

            axes = axis if isinstance(axis, tuple) else (axis,)
            axes = tuple(a if a >= 0 else a + self.data.ndim for a in axes)

            if not keepdims:
                for ax in sorted(axes):
                    grad = np.expand_dims(grad, axis=ax)

            self.grad += np.ones_like(self.data) * grad

        out._backward = _backward
        return out

    def mean(self, axis=None, keepdims=False):
        if axis is None:
            denom = self.data.size
        else:
            axes = axis if isinstance(axis, tuple) else (axis,)
            denom = np.prod([self.data.shape[a] for a in axes])

        return self.sum(axis=axis, keepdims=keepdims) * (1.0 / denom)

    def relu(self):
        out = Tensor(
            np.maximum(0.0, self.data),
            requires_grad=self.requires_grad,
            _children=(self,),
            _op="relu",
        )

        def _backward():
            if self.requires_grad:
                self.grad += out.grad * (self.data > 0.0)

        out._backward = _backward
        return out

    def tanh(self):
        t = np.tanh(self.data)
        out = Tensor(t, requires_grad=self.requires_grad, _children=(self,), _op="tanh")

        def _backward():
            if self.requires_grad:
                self.grad += out.grad * (1.0 - t ** 2)

        out._backward = _backward
        return out

    def backward(self):
        if self.data.size != 1:
            raise ValueError("`backward()` ist hier nur für skalare Ausgaben implementiert.")

        topo = []
        visited = set()

        def build(node):
            if node not in visited:
                visited.add(node)
                for parent in node._prev:
                    build(parent)
                topo.append(node)

        build(self)

        self.grad = np.ones_like(self.data)

        for node in reversed(topo):
            node._backward()

## Ein erster sehr kleiner Test

Wir prüfen die Ableitung von

$$
f(x) = x^2 + 3x + 1.
$$

Von Hand gilt

$$
f'(x) = 2x + 3.
$$

An der Stelle $x=2$ muss also $f'(2)=7$ herauskommen.

In [ ]:
x = Tensor([2.0], requires_grad=True)
f = x.pow(2) + 3.0 * x + 1.0
f.backward()

print("f(x) =", f.data)
print("berechneter Gradient =", x.grad)
print("erwarteter Gradient   =", 2 * 2.0 + 3.0)

Wenn hier $7$ herauskommt, ist das ein gutes erstes Signal.  
Aber für ein neuronales Netz reicht ein einzelner Test noch nicht.

Darum machen wir jetzt etwas Strengeres: einen **Gradientencheck** mit finiten Differenzen.


## Vom Rechengraphen zur Rückwärtsrechnung

Unsere Klasse `Tensor` speichert zu jeder Größe

- ihren aktuellen Zahlenwert,
- ihre Vorgänger im Rechengraphen,
- und eine lokale Rückwärtsfunktion.

Wenn zum Beispiel
$$
c = a+b
$$
gilt, dann ist lokal klar:
$$
\frac{\partial c}{\partial a}=1,
\qquad
\frac{\partial c}{\partial b}=1.
$$

Wenn dagegen
$$
c = a\cdot b,
$$
dann gilt
$$
\frac{\partial c}{\partial a}=b,
\qquad
\frac{\partial c}{\partial b}=a.
$$

Im Rückwärtslauf wird genau dieses lokale Wissen mit der Kettenregel kombiniert.  
Wird eine Variable mehrfach verwendet, müssen die Beiträge addiert werden. Auch das ist mathematisch korrekt, denn Ableitungen sind linear.


## Bausteine für ein kleines neuronales Netz

Wir definieren nun einfache Schichten und Container:

- `Linear`: affine Abbildung $x \mapsto xW + b$
- `Tanh` und `ReLU`: Aktivierungsfunktionen
- `Sequential`: führt mehrere Schichten nacheinander aus

Für dieses Notebook benutzen wir hauptsächlich `Tanh`, weil unsere Zielfunktion $\sin(x)$ sowohl positive als auch negative Werte annimmt.


## Wichtige Ableitungen für lineare Schichten

Für neuronale Netze ist die Operation
$$
Z = XW + b
$$
besonders wichtig.

Hier sind die Größen typischerweise so geformt:

- $X \in \mathbb{R}^{m\times d}$: ein Mini-Batch mit $m$ Datenpunkten,
- $W \in \mathbb{R}^{d\times p}$: Gewichte,
- $b \in \mathbb{R}^{1\times p}$: Bias,
- $Z \in \mathbb{R}^{m\times p}$: Ausgabe der linearen Schicht.

Angenommen, aus späteren Rechenschritten kommt bereits der Gradient
$$
G = \frac{\partial L}{\partial Z}.
$$
Dann lauten die Rückwärtsformeln:
$$
\frac{\partial L}{\partial W}=X^\top G,
\qquad
\frac{\partial L}{\partial b}=\sum_{i=1}^m G_{i,:},
\qquad
\frac{\partial L}{\partial X}=GW^\top.
$$

Diese drei Formeln tauchen im Code direkt wieder auf.

Der Bias ist hier besonders lehrreich:  
Im Vorwärtslauf wird $b$ per **Broadcasting** zu jeder Zeile von $XW$ addiert.  
Darum müssen wir im Rückwärtslauf die Gradienten über die Batchrichtung aufsummieren.


In [ ]:
class Module:
    def parameters(self):
        return []

    def zero_grad(self):
        for p in self.parameters():
            p.zero_grad()


class Linear(Module):
    def __init__(self, in_features, out_features):
        # Xavier-Initialisierung: gut geeignet für tanh-Netze
        limit = np.sqrt(6.0 / (in_features + out_features))
        self.weight = Tensor(
            np.random.uniform(-limit, limit, size=(in_features, out_features)),
            requires_grad=True,
        )
        self.bias = Tensor(np.zeros((1, out_features)), requires_grad=True)

    def __call__(self, x):
        return x @ self.weight + self.bias

    def parameters(self):
        return [self.weight, self.bias]


class Tanh(Module):
    def __call__(self, x):
        return x.tanh()


class ReLU(Module):
    def __call__(self, x):
        return x.relu()


class Sequential(Module):
    def __init__(self, *layers):
        self.layers = list(layers)

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        params = []
        for layer in self.layers:
            if isinstance(layer, Module):
                params.extend(layer.parameters())
        return params

## Verlustfunktion und Optimierer

Für die Regression nehmen wir den mittleren quadratischen Fehler

$$
\mathrm{MSE} = \frac{1}{N}\sum_{i=1}^N (\hat y_i - y_i)^2.
$$

Außerdem verwenden wir **stochastischen Gradientenabstieg** (SGD).

Das ist nicht der modernste Optimierer, aber didaktisch sehr gut:
Man sieht direkt, dass jeder Parameterschritt einfach in Richtung des negativen Gradienten geht.


## Die Verlustfunktion

Für Regression verwenden wir hier den mittleren quadratischen Fehler:
$$
L(\widehat{Y},Y)
=
\frac{1}{m}\sum_{i=1}^m (\widehat{y}_i-y_i)^2.
$$

Für einen einzelnen Datenpunkt lautet die Ableitung nach der Vorhersage
$$
\frac{\partial L}{\partial \widehat{y}}
=
\frac{2}{m}(\widehat{y}-y).
$$

Im Code schreiben wir die Verlustfunktion als Mittelwert von Quadraten.  
Die Ableitung entsteht dann automatisch aus den bereits implementierten Bausteinen:

1. Subtraktion,
2. Quadrieren,
3. Mittelwertbildung.


In [ ]:
def mse_loss(prediction, target):
    return ((prediction - target).pow(2)).mean()


class SGD:
    def __init__(self, parameters, lr=0.01):
        self.parameters = list(parameters)
        self.lr = lr

    def zero_grad(self):
        for p in self.parameters:
            p.zero_grad()

    def step(self):
        for p in self.parameters:
            p.data -= self.lr * p.grad

## Gradiententest mit finiten Differenzen

Bevor wir trainieren, prüfen wir die Korrektheit unserer Gradienten numerisch.

Idee:

1. Wir berechnen den Gradienten mit `backward()`.
2. Wir stören einen Parameter $w$ leicht nach oben und unten.
3. Wir approximieren die Ableitung mit

$$
\frac{L(w+\varepsilon)-L(w-\varepsilon)}{2\varepsilon}.
$$

Wenn beide Werte fast gleich sind, ist das ein starkes Indiz dafür, dass die Implementierung korrekt ist.

In [ ]:
np.random.seed(1)

test_layer = Linear(2, 1)
x_test = Tensor([[1.0, -2.0]])
y_test = Tensor([[0.5]])

loss = mse_loss(test_layer(x_test), y_test)
test_layer.zero_grad()
loss.backward()

autograd_grad = test_layer.weight.grad.copy()

epsilon = 1e-6
finite_diff_grad = np.zeros_like(test_layer.weight.data)

for i in range(test_layer.weight.data.shape[0]):
    for j in range(test_layer.weight.data.shape[1]):
        old_value = test_layer.weight.data[i, j]

        test_layer.weight.data[i, j] = old_value + epsilon
        loss_plus = mse_loss(test_layer(x_test), y_test).data.item()

        test_layer.weight.data[i, j] = old_value - epsilon
        loss_minus = mse_loss(test_layer(x_test), y_test).data.item()

        finite_diff_grad[i, j] = (loss_plus - loss_minus) / (2 * epsilon)
        test_layer.weight.data[i, j] = old_value

print("Gradient aus backward():")
print(autograd_grad)

print("\nGradient aus finiten Differenzen:")
print(finite_diff_grad)

print("\nMaximaler absoluter Fehler:")
print(np.max(np.abs(autograd_grad - finite_diff_grad)))

Ein sehr kleiner Fehler ist hier genau das, was wir sehen wollen.

Damit haben wir eine wesentlich bessere Grundlage als bei einer bloß optisch funktionierenden Trainingskurve.

## Das eigentliche Netz

Wir wählen ein kleines voll verbundenes Netz:

- Eingabegröße: 1
- zwei versteckte Schichten mit je 32 Neuronen
- `tanh` als Aktivierung
- Ausgabegröße: 1

Das ist klein genug zum Verstehen und groß genug, um $\sin(x)$ gut zu approximieren.

In [ ]:
np.random.seed(42)

model = Sequential(
    Linear(1, 32),
    Tanh(),
    Linear(32, 32),
    Tanh(),
    Linear(32, 1),
)

optimizer = SGD(model.parameters(), lr=0.05)

## Training

In jeder Epoche passiert Folgendes:

1. Wir mischen die Trainingsdaten.
2. Wir teilen sie in kleine Mini-Batches.
3. Für jedes Batch:
   - Vorwärtslauf
   - Verlust berechnen
   - Rückwärtslauf (`backward`)
   - Parameterupdate

Zusätzlich speichern wir Trainings- und Validierungsfehler.


## Gradientenverfahren und Mini-Batches

Sobald wir den Gradienten aller Parameter haben, führen wir ein Optimierungsschritt aus:
$$
\theta^{(k+1)} = \theta^{(k)} - \eta \nabla_\theta L(\theta^{(k)}).
$$

Im Notebook verwenden wir **stochastisches Gradientenverfahren** mit Mini-Batches:

- Die Trainingsdaten werden gemischt.
- Wir betrachten immer nur kleine Teilmengen.
- Für jedes Mini-Batch berechnen wir Vorwärtslauf, Verlust, Rückwärtslauf und Update.

Das ist numerisch und praktisch wichtig:

- Es spart Speicher,
- ist oft schneller,
- und führt in vielen Anwendungen zu gutem Lernverhalten.

Die Lernrate $\eta$ ist dabei ein zentraler Parameter.  
Ist sie zu groß, wird das Verfahren instabil.  
Ist sie zu klein, lernt das Modell nur sehr langsam.


In [ ]:
num_epochs = 2000
batch_size = 32

train_losses = []
val_losses = []

for epoch in range(num_epochs):
    permutation = np.random.permutation(len(x_train))
    epoch_loss = 0.0

    for start in range(0, len(x_train), batch_size):
        batch_idx = permutation[start:start + batch_size]

        xb = Tensor(x_train[batch_idx])
        yb = Tensor(y_train[batch_idx])

        prediction = model(xb)
        loss = mse_loss(prediction, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.data.item()

    train_loss = epoch_loss / int(np.ceil(len(x_train) / batch_size))
    val_prediction = model(Tensor(x_val))
    val_loss = mse_loss(val_prediction, Tensor(y_val)).data.item()

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if epoch % 200 == 0:
        print(f"Epoche {epoch:4d}: Trainingsfehler = {train_loss:.6f}, Validierungsfehler = {val_loss:.6f}")

## Lernkurven

Wenn alles korrekt implementiert ist, sollten beide Fehler im Verlauf deutlich kleiner werden.


## Was wir im Training beobachten wollen

Während des Trainings speichern wir den Fehler auf den Trainingsdaten und auf einem kleinen Validierungssatz.

Dabei interessieren uns vor allem drei Fragen:

1. **Fällt der Trainingsfehler?**  
   Dann arbeitet das Optimierungsverfahren grundsätzlich.

2. **Fällt auch der Validierungsfehler?**  
   Dann lernt das Netz nicht nur die Trainingsdaten auswendig, sondern generalisiert zumindest etwas.

3. **Bleiben beide Fehler auf ähnlichem Niveau?**  
   Große Unterschiede können ein Hinweis auf Überanpassung sein.


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="Training")
plt.plot(val_losses, label="Validierung")
plt.yscale("log")
plt.xlabel("Epoche")
plt.ylabel("MSE")
plt.title("Lernkurven")
plt.legend()
plt.show()

## Ergebnis: Wie gut approximiert das Netz die Funktion?

Jetzt vergleichen wir die wahre Funktion $\sin(x)$ mit der Vorhersage des trainierten Netzes.

In [ ]:
x_plot = np.linspace(-np.pi, np.pi, 400).reshape(-1, 1)
y_true = np.sin(x_plot)
y_pred = model(Tensor(x_plot)).data

plt.figure(figsize=(7, 4))
plt.plot(x_plot, y_true, label="sin(x)", linewidth=2)
plt.plot(x_plot, y_pred, "--", label="Netzvorhersage", linewidth=2)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Approximation von sin(x)")
plt.legend()
plt.show()


## Zusammenfassung

In diesem Notebook haben wir zwei Ebenen zusammengeführt:

### 1. Die mathematische Ebene
Wir haben das Training eines neuronalen Netzes als Optimierungsproblem formuliert:
$$
\min_\theta \frac{1}{N}\sum_{i=1}^N \ell\bigl(F(x_i;\theta),y_i\bigr).
$$
Die dafür nötigen Gradienten entstehen aus der Kettenregel.  
Für lineare Schichten $XW+b$, Aktivierungsfunktionen und die MSE-Verlustfunktion lassen sich die Rückwärtsformeln explizit angeben.

### 2. Die algorithmische Ebene
Wir haben eine kleine `Tensor`-Klasse gebaut, die

- Werte speichert,
- einen Rechengraphen aufbaut,
- und im Rückwärtslauf Gradienten automatisch berechnet.

Damit entsteht aus vielen kleinen lokalen Ableitungen automatisch der gesamte Gradient des Netzes.

### Was besonders wichtig ist
- Backpropagation ist keine „magische“ Spezialtechnik, sondern systematische Anwendung der Kettenregel.
- Broadcasting im Vorwärtslauf führt zu Summen im Rückwärtslauf.
- Ein Gradiententest mit finiten Differenzen ist ein sehr gutes Werkzeug, um Implementierungen zu prüfen.
- Auch ein kleines selbst geschriebenes System kann bereits ein neuronales Netz trainieren.

Wenn Sie dieses Notebook verstanden haben, dann kennen Sie nicht nur die Benutzung eines Frameworks, sondern auch die mathematische Idee dahinter.
